# Experiment 36 — From-scratch PheromoneWalker on Beauty

This experiment starts from the **corrected SparseWalker v1.1 Amazon architecture** but uses **no pretrained checkpoint, no optimizer, and no backward pass**.

Item identity is assigned a unique concept address without using behavior data. Sequential structure is then learned only through ACO-style pheromone deposition, evaporation, and slow once-per-epoch sparse rewiring.

Reference Beauty test NDCG@10: SASRec FullCE **0.03120**; gradient-trained SparseWalker v1.1 **0.04488**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, json, runpy, torch
from pathlib import Path

REPO='/content/Sparsewalker'
BRANCH='agent/pheromone-walker-v1'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO], check=True)
for p in [f'{REPO}/src', f'{REPO}/experiments', f'{REPO}/benchmarks']:
    if p not in sys.path: sys.path.insert(0,p)
import sparsewalker
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__,'bf16',torch.cuda.is_bf16_supported())
print('BRANCH',BRANCH,'PACKAGE',sparsewalker.__file__)


## Run Beauty

Watch `PHERO_INIT` first. It must report `unique_item_concept_addresses == n_items` and `router_target_hit_rate >= 0.99`; otherwise the run aborts before training.

The graph starts random. Pheromone learning is intentionally slow: only 5% of evidence-supported sources may rewire one non-self edge per epoch.


In [ ]:
SCRIPT=f'{REPO}/experiments/run_amazon_pheromone_walker.py'
sys.argv=[SCRIPT,
    '--dataset','beauty',
    '--epochs','30',
    '--batch-size','512',
    '--eval-every','1',
    '--pheromone-beta','2.0',
    '--deposit-lr','0.5',
    '--evaporation','0.10',
    '--rewire-fraction','0.05',
    '--new-edge-tau','2.0',
    '--message-gain','16.0',
]
runpy.run_path(SCRIPT, run_name='__main__')


## Result summary

The first question is simply whether NDCG rises decisively above the random initialization. Crossing SASRec (`0.03120`) would be a major result; approaching gradient Walker (`0.04488`) would be exceptional.


In [ ]:
p=Path('/content/drive/MyDrive/sparsewalker_pheromone_from_scratch/beauty/seed42/result.json')
if p.exists():
    r=json.loads(p.read_text())
    print(json.dumps(r, indent=2))
    n=r['best_pheromone']['test']['NDCG@10']
    print('\nBeauty test NDCG@10')
    print('PheromoneWalker:',n)
    print('SASRec:',0.031195719394901355,'ratio',n/0.031195719394901355)
    print('Gradient SparseWalker:',0.044882819399656555,'ratio',n/0.044882819399656555)
else:
    print('Run the training cell first.')


## What is and is not learned

**No warm start.** No sequence-trained neural weights are loaded. The item→concept mapping is a deterministic unique address code using no behavior data. The sequential knowledge comes from pheromone and topology updates only. The recurrent state evolution, K=8 pruning, factorized concept space, degree-4 graph, two hops, and readout all remain the corrected SparseWalker v1.1 architecture.
